# Model Experimentation

## Objective

The purpose of this notebook is to evaluate multiple supervised machine learning algorithms for predicting employee attrition.

Each model will be trained using the same preprocessing pipeline and evaluated using consistent performance metrics.

The goal is to identify the model that provides the best balance between predictive performance, robustness, computational efficiency, and interpretability.

In [14]:
# =============================================================================
# Import Required Libraries
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

# Visualization
import plotly.express as px
import plotly.graph_objects as go

# Data Splitting
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [15]:
# =============================================================================
# 2. Load Dataset
# =============================================================================

DATA_PATH = Path("../dataset/employee_attrition.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [16]:
# =============================================================================
# Remove Constant & Identifier Columns
# =============================================================================

columns_to_drop = [
    "EmployeeCount",
    "EmployeeNumber",
    "Over18",
    "StandardHours"
]

df = df.drop(columns=columns_to_drop)

print(df.shape)

(1470, 31)


In [17]:
# =============================================================================
# Features and Target
# =============================================================================

X = df.drop(columns="Attrition")

y = df["Attrition"].map({
    "No":0,
    "Yes":1
})

print(X.shape)
print(y.shape)

(1470, 30)
(1470,)


In [18]:
# =============================================================================
# Train Test Split
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training Shape :", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape : (1176, 30)
Testing Shape : (294, 30)


In [19]:
# =============================================================================
# Numerical & Categorical Features
# =============================================================================

numerical_features = X_train.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical :", len(numerical_features))
print("Categorical :", len(categorical_features))

Numerical : 23
Categorical : 7


In [20]:
# =============================================================================
# Numerical Pipeline
# =============================================================================

numerical_pipeline = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# =============================================================================
# Categorical Pipeline
# =============================================================================

categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

# =============================================================================
# Column Transformer
# =============================================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            numerical_pipeline,
            numerical_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [21]:
# =============================================================================
# Fit & Transform
# =============================================================================

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(1176, 51)
(294, 51)


In [22]:
# =============================================================================
# Import Machine Learning Models
# =============================================================================

from sklearn.linear_model import LogisticRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    ExtraTreesClassifier
)

from sklearn.naive_bayes import GaussianNB

from sklearn.neighbors import KNeighborsClassifier

from sklearn.svm import SVC

from xgboost import XGBClassifier

from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

In [23]:
# =============================================================================
# Baseline Models
# =============================================================================

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=42
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        ),

    "AdaBoost":
        AdaBoostClassifier(
            random_state=42
        ),

    "Gaussian Naive Bayes":
        GaussianNB(),

    "K-Nearest Neighbors":
        KNeighborsClassifier(),

    "Support Vector Machine":
        SVC(
            probability=True,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            random_state=42,
            eval_metric="logloss"
        ),

    "LightGBM":
        LGBMClassifier(
            random_state=42
        ),

    "CatBoost":
        CatBoostClassifier(
            verbose=False,
            random_state=42
        )

}

In [24]:
# =============================================================================
# Model Evaluation Function
# =============================================================================

def evaluate_model(model, X_train, y_train, X_test, y_test):

    # Train model
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Prediction probabilities
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {

        "Accuracy":
            accuracy_score(y_test, y_pred),

        "Precision":
            precision_score(y_test, y_pred),

        "Recall":
            recall_score(y_test, y_pred),

        "F1 Score":
            f1_score(y_test, y_pred),

        "ROC AUC":
            roc_auc_score(y_test, y_prob)

    }

    return metrics

In [25]:
# =============================================================================
# Train All Models
# =============================================================================

results = {}

for model_name, model in models.items():

    print(f"Training {model_name}...")

    results[model_name] = evaluate_model(
        model,
        X_train_processed,
        y_train,
        X_test_processed,
        y_test
    )

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training Extra Trees...
Training Gradient Boosting...
Training AdaBoost...
Training Gaussian Naive Bayes...
Training K-Nearest Neighbors...
Training Support Vector Machine...
Training XGBoost...
Training LightGBM...
[LightGBM] [Info] Number of positive: 190, number of negative: 986
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001968 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1170
[LightGBM] [Info] Number of data points in the train set: 1176, number of used features: 51
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.161565 -> initscore=-1.646632
[LightGBM] [Info] Start training from score -1.646632
Training CatBoost...


In [26]:
# =============================================================================
# Model Comparison
# =============================================================================

results_df = pd.DataFrame(results).T

results_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.860544,0.615385,0.340426,0.438356,0.811526
Decision Tree,0.765306,0.310345,0.382979,0.342857,0.610518
Random Forest,0.850340,0.636364,0.148936,0.241379,0.786976
Extra Trees,0.863946,0.705882,0.255319,0.375000,0.799208
Gradient Boosting,0.860544,0.714286,0.212766,0.327869,0.805668
AdaBoost,0.829932,0.440000,0.234043,0.305556,0.799552
Gaussian Naive Bayes,0.646259,0.260504,0.659574,0.373494,0.703161
K-Nearest Neighbors,0.843537,0.538462,0.148936,0.233333,0.594625
Support Vector Machine,0.860544,0.750000,0.191489,0.305085,0.816177
XGBoost,0.860544,0.650000,0.276596,0.388060,0.738220


In [27]:
# =============================================================================
# Sort Models
# =============================================================================

results_df = results_df.sort_values(
    by="ROC AUC",
    ascending=False
)

results_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Support Vector Machine,0.860544,0.750000,0.191489,0.305085,0.816177
Logistic Regression,0.860544,0.615385,0.340426,0.438356,0.811526
Gradient Boosting,0.860544,0.714286,0.212766,0.327869,0.805668
CatBoost,0.857143,0.692308,0.191489,0.300000,0.804290
AdaBoost,0.829932,0.440000,0.234043,0.305556,0.799552
Extra Trees,0.863946,0.705882,0.255319,0.375000,0.799208
Random Forest,0.850340,0.636364,0.148936,0.241379,0.786976
LightGBM,0.860544,0.687500,0.234043,0.349206,0.767680
XGBoost,0.860544,0.650000,0.276596,0.388060,0.738220
Gaussian Naive Bayes,0.646259,0.260504,0.659574,0.373494,0.703161


In [28]:
# =============================================================================
# Model Performance Summary
# =============================================================================

results_df = results_df.round(4)

results_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Support Vector Machine,0.8605,0.7500,0.1915,0.3051,0.8162
Logistic Regression,0.8605,0.6154,0.3404,0.4384,0.8115
Gradient Boosting,0.8605,0.7143,0.2128,0.3279,0.8057
CatBoost,0.8571,0.6923,0.1915,0.3000,0.8043
AdaBoost,0.8299,0.4400,0.2340,0.3056,0.7996
Extra Trees,0.8639,0.7059,0.2553,0.3750,0.7992
Random Forest,0.8503,0.6364,0.1489,0.2414,0.7870
LightGBM,0.8605,0.6875,0.2340,0.3492,0.7677
XGBoost,0.8605,0.6500,0.2766,0.3881,0.7382
Gaussian Naive Bayes,0.6463,0.2605,0.6596,0.3735,0.7032


In [29]:
# =============================================================================
# ROC-AUC Comparison
# =============================================================================

fig = px.bar(
    results_df.reset_index(),
    x="index",
    y="ROC AUC",
    color="ROC AUC",
    text="ROC AUC",
    title="ROC-AUC Comparison of Baseline Models",
)

fig.update_traces(texttemplate="%{text:.3f}")

fig.update_layout(
    xaxis_title="Model",
    yaxis_title="ROC-AUC",
    xaxis_tickangle=-30
)

fig.show()

In [30]:
metrics = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"
]

comparison_df = results_df[metrics]

comparison_df

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Support Vector Machine,0.8605,0.7500,0.1915,0.3051,0.8162
Logistic Regression,0.8605,0.6154,0.3404,0.4384,0.8115
Gradient Boosting,0.8605,0.7143,0.2128,0.3279,0.8057
CatBoost,0.8571,0.6923,0.1915,0.3000,0.8043
AdaBoost,0.8299,0.4400,0.2340,0.3056,0.7996
Extra Trees,0.8639,0.7059,0.2553,0.3750,0.7992
Random Forest,0.8503,0.6364,0.1489,0.2414,0.7870
LightGBM,0.8605,0.6875,0.2340,0.3492,0.7677
XGBoost,0.8605,0.6500,0.2766,0.3881,0.7382
Gaussian Naive Bayes,0.6463,0.2605,0.6596,0.3735,0.7032


In [31]:
fig = px.imshow(
    comparison_df,
    text_auto=".3f",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Model Performance Across Evaluation Metrics"
)

fig.show()

In [32]:
top_models = results_df.head(3)

top_models

,Accuracy,Precision,Recall,F1 Score,ROC AUC
Support Vector Machine,0.8605,0.7500,0.1915,0.3051,0.8162
Logistic Regression,0.8605,0.6154,0.3404,0.4384,0.8115
Gradient Boosting,0.8605,0.7143,0.2128,0.3279,0.8057


# Conclusion

In this notebook, twelve baseline machine learning algorithms were trained and evaluated using a common preprocessing pipeline. Model performance was assessed using Accuracy, Precision, Recall, F1-score, and ROC-AUC to provide a comprehensive comparison.

Although several models achieved similar accuracy, the evaluation showed that accuracy alone is not an appropriate metric for this imbalanced classification problem. Greater emphasis was placed on Recall, F1-score, and ROC-AUC to assess each model's ability to identify employees who are likely to leave.

Among all baseline models, **Logistic Regression** demonstrated the best overall performance by achieving the highest F1-score (0.4384) and the highest Recall (0.3404) among the top-performing models, while maintaining a competitive ROC-AUC (0.8115). These results indicate that Logistic Regression provides the best balance between identifying employees at risk of attrition and minimizing incorrect predictions in the baseline experiments.

Support Vector Machine achieved the highest ROC-AUC (0.8162) and Precision (0.7500), indicating strong class discrimination. However, its low Recall suggests that it failed to identify many employees who actually left the organization. XGBoost emerged as the strongest boosting algorithm in the baseline comparison, making it a suitable candidate for further optimization.

Based on the baseline evaluation, the following models have been shortlisted for hyperparameter tuning:

- Logistic Regression
- Support Vector Machine
- XGBoost

The final production model will be selected only after hyperparameter tuning and further evaluation in the next notebook.